In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kalyandath/deberta-pretrained/config.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/training_args.bin
/kaggle/input/datasets/kalyandath/deberta-pretrained/tokenizer.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/tokenizer_config.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/model.safetensors
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [15]:
sample_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
print(sample_df.head())

   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C


In [16]:
# sample_df.to_csv("submission.csv", index = False)

In [17]:
MODEL = "/kaggle/input/datasets/kalyandath/deberta-pretrained/"

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForMultipleChoice.from_pretrained(MODEL)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [19]:
## tokenization
def preprocess(data):
    output = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    label_map = {'A':0, 'B':1,'C':2, 'D':3, 'E':4}

    prompts = []
    options = []
    answer = []

    for i in range(len(data["prompt"])):
        prompts += [data["prompt"][i]] * 5
        options += [data[x][i] for x in label_map.keys()]
        answer.append(label_map[data["answer"][i]])

    tokenized_input = tokenizer(prompts, options, truncation = True, max_length = 256)
    
    for i in range(0, len(tokenized_input["input_ids"]),5):
        tokenized_return = {
            "input_ids": [],
            "attention_mask": [],
        }

        tokenized_return["input_ids"] += tokenized_input["input_ids"][i:i+5]
        tokenized_return["attention_mask"] += tokenized_input["attention_mask"][i:i+5]
        tokenized_return["labels"] = answer[i//5]

        output["input_ids"].append(tokenized_return["input_ids"])
        output["attention_mask"].append(tokenized_return["attention_mask"])
        output["labels"].append(tokenized_return["labels"])

    print(output["input_ids"][0])

    return output

In [20]:
## Load the test csv
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test_df["answer"] = "A"
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[[1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 262, 454, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 1026, 843, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 261, 359, 1940, 92183, 303, 266, 5786, 53930, 6642, 275, 266, 467, 5465, 260, 2], [1, 8498, 262, 410, 628, 1330, 294, 458, 269, 262, 1328, 457, 262, 92183, 268, 263, 53930, 36856, 267, 1850, 82785, 10272, 9143, 302, 3153, 260, 2, 434, 469, 53930, 6642, 265, 311, 92183, 26

In [23]:
from transformers import DataCollatorForMultipleChoice

data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
inference_trainer = Trainer(
    model = model,
    data_collator = data_collator
)

print("Running Predictions")
predictions = inference_trainer.predict(tokenized_test)
logits = predictions.predictions

Running Predictions


In [25]:
top_3_indices = np.argsort(logits, axis=1)[:,-1:-4:-1]
index_to_letter = {0:"A", 1:"B", 2:"C", 3:"D", 4: "E"}
predicted_strings = []

for row in top_3_indices:
    letters = [index_to_letter[idx] for idx in row]
    predicted_strings.append(" ".join(letters))

submission_df = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predicted_strings
})

In [26]:
# Submission for pretrained model
submission_df.to_csv("submission.csv", index= False)
print(submission_df.head())

   id prediction
0   1      A D B
1   2      B D A
2   3      B E D
3   4      E C A
4   5      C D A
